# AI Catalog Studio — Google Colab
Хмарний запуск тієї самої готової моделі Qwen3 4B Instruct. За можливості оберіть GPU в Runtime → Change runtime type. Генерація використовує engine.py з проєкту; реальні результати потрібно перевірити перед публікацією. Це не навчання моделі з нуля.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, sys
uploaded = files.upload()  # оберіть AI_Catalog_Studio.zip
name = next(iter(uploaded))
root = Path('/content/catalog-studio')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(name) as z:
    z.extractall(root)
project = next(root.rglob('engine.py')).parent
sys.path.insert(0, str(project))
print(project)

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, urllib.request
server = subprocess.Popen(['ollama','serve'], stdout=open('/tmp/ollama.log','w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags',timeout=2).close()
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Ollama не запустився. Перевір /tmp/ollama.log')
subprocess.run(['ollama','pull','qwen3:4b-instruct'],check=True)
subprocess.run(['ollama','list'],check=True)

In [ ]:
import json, engine
products = engine.validate_products(json.loads((project/'sample-products.json').read_text(encoding='utf-8')))
categories = engine.generate_categories(products)
print('Model:',engine.MODEL)
print(json.dumps(categories, ensure_ascii=False, indent=2))
cards = {}
for p in products:
    print('Generating:',p['name'])
    cards[p['sku']] = engine.generate_card(p,categories)
    print(json.dumps(cards[p['sku']],ensure_ascii=False,indent=2))

## Перевірка та експорт
Перевір усі факти перед публікацією. Виправ помилкові тексти у словнику `cards`, якщо потрібно. Після перевірки явно встанови `APPROVE = True`.

In [ ]:
APPROVE = False
if not APPROVE:
    raise ValueError('Перевір контент і встанови APPROVE = True.')
output = engine.export_catalog(products,categories,cards)
engine.save_json('/content/catalog.json',output)
files.download('/content/catalog.json')